In [39]:
import polars as pl

In [40]:
#fact = pl.read_parquet("../data/gold/fact_trafico_hora.parquet")
#fecha = pl.read_parquet("../data/gold/dim_fecha.parquet")
#sensor = pl.read_parquet("../data/gold/dim_sensor.parquet")

In [41]:
fact = pl.scan_parquet("../data/gold/fact_trafico_hora.parquet")
fecha = pl.scan_parquet("../data/gold/dim_fecha.parquet")
sensor = pl.scan_parquet("../data/gold/dim_sensor.parquet")

In [42]:
#print(f"Fact: {fact.shape}")
#print(f"Fecha: {fecha.shape}")
#print(f"Sensor: {sensor.shape}")

In [43]:
# Reemplaza la celda 7 por esto:
print(f"Fact: ({fact.select(pl.len()).collect().item()}, {len(fact.columns)})")
print(f"Fecha: ({fecha.select(pl.len()).collect().item()}, {len(fecha.columns)})")
print(f"Sensor: ({sensor.select(pl.len()).collect().item()}, {len(sensor.columns)})")

Fact: (40521393, 13)
Fecha: (365, 8)
Sensor: (5088, 9)


C:\Users\Usuario\AppData\Local\Temp\ipykernel_11744\1873667190.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(f"Fact: ({fact.select(pl.len()).collect().item()}, {len(fact.columns)})")
C:\Users\Usuario\AppData\Local\Temp\ipykernel_11744\1873667190.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(f"Fecha: ({fecha.select(pl.len()).collect().item()}, {len(fecha.columns)})")
C:\Users\Usuario\AppData\Local\Temp\ipykernel_11744\1873667190.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().nam

In [44]:
fact.schema

C:\Users\Usuario\AppData\Local\Temp\ipykernel_11744\4225466638.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  fact.schema


Schema([('id_sensor', Int32),
        ('id_fecha', Date),
        ('hora', Int32),
        ('intensidad_media', Float64),
        ('intensidad_max', Float64),
        ('intensidad_min', Float64),
        ('ocupacion_media', Float64),
        ('ocupacion_max', Float64),
        ('velocidad_media', Float64),
        ('velocidad_min', Float64),
        ('num_mediciones', Int64),
        ('num_error_E', Float64),
        ('porcentaje_calidad', Float64)])

In [45]:
fecha.schema

C:\Users\Usuario\AppData\Local\Temp\ipykernel_11744\2622028573.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  fecha.schema


Schema([('id_fecha', Date),
        ('año', Int64),
        ('mes', Int64),
        ('nombre_mes', String),
        ('trimestre', Int64),
        ('dia', Int64),
        ('dia_semana', Int64),
        ('fin_semana', Boolean)])

In [46]:
sensor.schema

C:\Users\Usuario\AppData\Local\Temp\ipykernel_11744\2693837341.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  sensor.schema


Schema([('id_sensor', Int32),
        ('tipo_elem', String),
        ('distrito', Int32),
        ('cod_cent', String),
        ('nombre_norm', String),
        ('utm_x', Float64),
        ('utm_y', Float64),
        ('latitud', Float64),
        ('longitud', Float64)])

In [47]:
#fact.head()
fact.head().collect()

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,ocupacion_media,ocupacion_max,velocidad_media,velocidad_min,num_mediciones,num_error_E,porcentaje_calidad
i32,date,i32,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64
1001,2026-04-18,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-21,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-21,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1002,2026-04-16,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1002,2026-04-17,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0


In [48]:
#fecha.head()
fecha.head().collect

<bound method LazyFrame.collect of <LazyFrame at 0x16B55051970>>

In [49]:
#sensor.head()
sensor.head().collect

<bound method LazyFrame.collect of <LazyFrame at 0x16B1463F050>>

In [50]:
#print(f"Sensores fact: {fact['id_sensor'].n_unique()}")
#print(f"Sensores dimensión: {sensor['id_sensor'].n_unique()}")

In [51]:
# Obtener valores únicos de forma perezosa y luego colectar el resultado
sensores_fact = fact.select(pl.col("id_sensor").n_unique()).collect().item()
sensores_dim = sensor.select(pl.col("id_sensor").n_unique()).collect().item()

print(f"Sensores fact: {sensores_fact}")
print(f"Sensores dimensión: {sensores_dim}")

Sensores fact: 4933
Sensores dimensión: 5088


In [52]:
#sensores_faltantes = (
#    set(fact["id_sensor"].unique())
#    - set(sensor["id_sensor"].unique())
#)

#print(len(sensores_faltantes))

In [53]:
# Calcular los sensores faltantes usando operaciones perezosas
sensores_fact_set = set(fact.select("id_sensor").unique().collect()["id_sensor"])
sensores_dim_set = set(sensor.select("id_sensor").unique().collect()["id_sensor"])

sensores_faltantes = sensores_fact_set - sensores_dim_set
print(len(sensores_faltantes))

0


In [54]:
df = (
    fact
    .join(fecha, on="id_fecha", how="left")
    .join(sensor, on="id_sensor", how="left")
)

In [55]:
#print(fact.height)
#print(df.height)

In [ ]:
# En lugar de print(fact.height) y print(df.height)
print(fact.select(pl.len()).collect().item())
print(df.select(pl.len()).collect().item())

40521393
40521393


In [ ]:
df.select(
    pl.col("año").is_null().sum()
)

In [ ]:
df.select(
    pl.col("tipo_elem").is_null().sum()
)

In [60]:
#clave_unica = df.select(
#    pl.struct(
#        ["id_sensor","id_fecha","hora"]
#    ).n_unique()
#).item()

#print(f"Filas: {df.height}")
#print(f"Claves únicas: {clave_unica}")

clave_unica = df.select(
    pl.struct(
        ["id_sensor", "id_fecha", "hora"]
    ).n_unique()
).collect().item()

total_filas = df.select(pl.len()).collect().item()

print(f"Filas: {total_filas}")
print(f"Claves únicas: {clave_unica}")

Filas: 40521393
Claves únicas: 40521393


In [61]:
#(
#    df.null_count()
#      .transpose(
#          include_header=True,
#          header_name="columna",
#          column_names=["nulos"]
#      )
#      .sort("nulos", descending=True)
#)
(
    df.null_count()
      .collect() # Traemos a memoria sólo la fila de conteo de nulos (operación muy ligera)
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""nombre_norm""",111090
"""distrito""",42529
"""velocidad_media""",33781
"""velocidad_min""",33781
"""id_sensor""",0
…,…
"""cod_cent""",0
"""utm_x""",0
"""utm_y""",0


In [62]:
df = df.drop(
    "nombre_norm",
    "cod_cent",
    "utm_x",
    "utm_y"
)

In [63]:
df = df.with_columns(
    pl.col("distrito")
      .fill_null(0)
      .cast(pl.Int32)
)

In [64]:
#df.group_by("tipo_elem").agg([
#    pl.col("velocidad_media")
#      .filter(pl.col("velocidad_media") > 0)
#      .len()
#      .alias("registros_con_velocidad")
#])

df.group_by("tipo_elem").agg([
    pl.col("velocidad_media")
      .filter(pl.col("velocidad_media") > 0)
      .len()
      .alias("registros_con_velocidad")
]).collect()

tipo_elem,registros_con_velocidad
str,u32
"""other""",762297
"""M30""",2361374
"""URB""",1


In [65]:
# 1. Tratamiento inicial de la velocidad

df = df.with_columns([

    pl.when(pl.col("tipo_elem") == "URB")
      .then(0)
      .when(pl.col("velocidad_media") < 0)
      .then(None)
      .otherwise(pl.col("velocidad_media"))
      .alias("velocidad_media"),

    pl.when(pl.col("tipo_elem") == "URB")
      .then(0)
      .when(pl.col("velocidad_min") < 0)
      .then(None)
      .otherwise(pl.col("velocidad_min"))
      .alias("velocidad_min")

])

In [66]:
# 2. Calcular la mediana por sensor

medianas = (
    df.group_by("id_sensor")
      .agg([
          pl.col("velocidad_media").median().alias("mediana_media"),
          pl.col("velocidad_min").median().alias("mediana_min")
      ])
)

df = df.join(medianas, on="id_sensor", how="left")

In [67]:
# 3. Imputar los nulos con la mediana del sensor

df = df.with_columns([

    pl.col("velocidad_media")
      .fill_null(pl.col("mediana_media"))
      .alias("velocidad_media"),

    pl.col("velocidad_min")
      .fill_null(pl.col("mediana_min"))
      .alias("velocidad_min")

]).drop(["mediana_media", "mediana_min"])

In [68]:
#df.select([
#    pl.col("velocidad_media").is_null().sum(),
#    pl.col("velocidad_min").is_null().sum()
#])

df.select([
    pl.col("velocidad_media").is_null().sum(),
    pl.col("velocidad_min").is_null().sum()
]).collect()

velocidad_media,velocidad_min
u32,u32
0,0


In [69]:
#(
#    df.null_count()
#      .transpose(
#          include_header=True,
#          header_name="columna",
#          column_names=["nulos"]
#      )
#      .sort("nulos", descending=True)
#)
(
    df.null_count()
      .collect()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""id_sensor""",0
"""id_fecha""",0
"""hora""",0
"""intensidad_media""",0
"""intensidad_max""",0
…,…
"""fin_semana""",0
"""tipo_elem""",0
"""distrito""",0


In [70]:
# Comprobar si existen registros con el valor 99999

df.filter(
    pl.col("intensidad_max") == 99999
).select(pl.len())

In [71]:
# Inspeccionar el registro antes de eliminarlo

df.filter(
    pl.col("intensidad_max") == 99999
).select([
    "id_sensor",
    "id_fecha",
    "hora",
    "intensidad_media",
    "intensidad_max",
    "intensidad_min",
    "num_mediciones"
])

In [72]:
# Analizar las intensidades más elevadas del conjunto de datos

df.filter(
    pl.col("intensidad_media") > 10000
).select([
    "id_sensor",
    "id_fecha",
    "hora",
    "intensidad_media",
    "intensidad_max",
    "intensidad_min",
    "num_mediciones"
]).sort(
    "intensidad_media",
    descending=True
)

In [73]:
# Eliminar únicamente el registro con el valor centinela 99999

df = df.filter(
    pl.col("intensidad_max") != 99999
)

In [74]:
#df.write_parquet("../data/modeling/base_modelado.parquet")

In [75]:
# Reemplaza la última celda por esto:
df.sink_parquet("../data/modeling/base_modelado.parquet")